In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

train_df = pd.read_csv('train.csv')

train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [2]:
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
train_df.groupby('Sex')['Survived'].mean()

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

In [4]:
train_df.pivot_table('Survived', index='Sex', columns='Pclass', aggfunc=['mean', 'std'])

mean                           std                    
Pclass         1         2         3         1         2         3
Sex                                                               
female  0.968085  0.921053  0.500000  0.176716  0.271448  0.501745
male    0.368852  0.157407  0.135447  0.484484  0.365882  0.342694

In [5]:
bins = [0, 12, 18, 35, 60, 80]
labels = ['Child', 'Teenager', 'Young Adult', 'Adult', 'Senior']

train_df['AgeGroup'] = pd.cut(train_df['Age'], bins=bins, labels=labels)

age_survival = train_df.pivot_table('Survived', index='AgeGroup', columns='Pclass')
age_survival

Pclass,1,2,3
AgeGroup,,,
Child,0.750000,1.000000,0.416667
Teenager,0.916667,0.500000,0.282609
Young Adult,0.757576,0.436170,0.232323
Adult,0.611111,0.382979,0.086207
Senior,0.214286,0.333333,0.200000


In [6]:
train_df['Title'] = train_df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

train_df['Title'].value_counts()

Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64

In [7]:
pd.crosstab(train_df['Title'], train_df['Sex'])

Sex,female,male
Title,,
Capt,0,1
Col,0,2
Countess,1,0
Don,0,1
Dr,1,6
Jonkheer,0,1
Lady,1,0
Major,0,2
Master,0,40


In [8]:
train_df['Title'] = train_df['Title'].replace(['Lady', 'Countess', 'Capt', 'Col',
        'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')

train_df['Title'] = train_df['Title'].replace('Mlle', 'Miss')
train_df['Title'] = train_df['Title'].replace('Ms', 'Miss')
train_df['Title'] = train_df['Title'].replace('Mme', 'Mrs')

train_df.groupby('Title')['Survived'].mean().sort_values(ascending=False)

Title
Mrs       0.793651
Miss      0.702703
Master    0.575000
Rare      0.347826
Mr        0.156673
Name: Survived, dtype: float64

In [9]:
titles_age_map = train_df.groupby('Title')['Age'].median()
train_df['Age'] = train_df['Age'].fillna(train_df.groupby('Title')['Age'].transform('median'))

In [10]:
bins = [0, 12, 18, 35, 60, 80]
labels = ['Child', 'Teenager', 'Young Adult', 'Adult', 'Senior']
train_df['AgeGroup'] = pd.cut(train_df['Age'], bins=bins, labels=labels)

final_age_df = train_df.pivot_table('Survived', index='AgeGroup', columns='Pclass')
final_age_df

Pclass,1,2,3
AgeGroup,,,
Child,0.750000,1.000000,0.423077
Teenager,0.916667,0.500000,0.282609
Young Adult,0.673684,0.428571,0.236364
Adult,0.604396,0.382979,0.086207
Senior,0.214286,0.333333,0.200000


In [11]:
columns_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'AgeGroup']
train_clean = train_df.drop(columns=columns_to_drop, errors='ignore')

train_clean['Embarked'] = train_clean['Embarked'].fillna(train_clean['Embarked'].mode()[0])
train_encoded = pd.get_dummies(train_clean, columns=['Sex', 'Embarked', 'Title'], drop_first=True)
train_encoded.shape

(891, 13)

In [12]:
train_encoded.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,3,22.0,1,0,7.2500,True,False,True,False,True,False,False
1,1,1,38.0,1,0,71.2833,False,False,False,False,False,True,False
2,1,3,26.0,0,0,7.9250,False,False,True,True,False,False,False
3,1,1,35.0,1,0,53.1000,False,False,True,False,False,True,False
4,0,3,35.0,0,0,8.0500,True,False,True,False,True,False,False


In [23]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import xgboost

X = train_encoded.drop('Survived', axis=1)
y = train_encoded['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_preds)

xgb = xgboost.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_preds)

In [24]:
rf_accuracy

0.7988826815642458

In [25]:
f1_score(y_test, rf_preds)

0.7313432835820896

In [26]:
xgb_accuracy

0.8044692737430168

In [27]:
f1_score(y_test, xgb_preds)

0.732824427480916

In [28]:
test_df = pd.read_csv('test.csv')
passenger_ids = test_df['PassengerId']

test_df['Title'] = test_df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test_df['Title'] = test_df['Title'].replace(['Lady', 'Countess', 'Capt', 'Col',
        'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')

test_df['Title'] = test_df['Title'].replace(['Mlle', 'Ms'], 'Miss')
test_df['Title'] = test_df['Title'].replace('Mme', 'Mrs')

test_df['Age'] = test_df['Age'].fillna(test_df.groupby('Title')['Age'].transform('median'))
test_df['Fare'] = test_df['Fare'].fillna(test_df['Fare'].median())

test_clean = test_df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'AgeGroup'], errors='ignore')

test_encoded = pd.get_dummies(test_clean, columns=['Sex', 'Embarked', 'Title'], drop_first=True)
test_encoded = test_encoded.reindex(columns=X.columns, fill_value=0)

final_predictions = xgb.predict(test_encoded)

submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': final_predictions
})

submission.to_csv('submission.csv', index=False)